# 五会话标签未形成精确单笔整股：订单形态审计

冻结后只选来源/路径成熟但 `no_exact_whole_share_pair` 的 50719 个分钟意向，按同一父本q/cash、Qlib完成bar和下一bar填单价重建SOXL/SOXX两臂订单。只查订单数、方向、符号、费用计数和保存funded标志；不选取或分析任何经济价值列。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
from copy import deepcopy
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.ibkr_tiered import settle_tiered_event
from src.analysis.stage1_capfree_lock_batch import _fee_snapshots
root=Path.cwd();card=root/'research/stage1_v5_no_exact_fill_order_audit_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
label_card=root/plan['label_card'];assert sha(label_card/'summary.json')==plan['label_summary_sha256'] and sha(label_card/'verification_manifest.json')==plan['label_manifest_sha256']
assert json.loads((label_card/'nas_binding.json').read_text())['evidence']['sha256']==plan['label_nas_archive_sha256']
p=plan['parent'];assert sha(root/p['card']/'summary.json')==p['summary_sha256'] and sha(root/p['card']/'verification_manifest.json')==p['manifest_sha256']
assert json.loads((root/p['card']/'nas_binding.json').read_text())['evidence']['sha256']==p['nas_archive_sha256']
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256'] and sha(root/plan['tariff_source_plan'])==plan['tariff_source_plan_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
objects,_=load_verified_recorder_objects(label_card,expected_recorder_id=plan['label_recorder_id'],artifacts=['analysis/labels','analysis/audit'],expected_manifest_sha256=plan['label_manifest_sha256'])
full=objects['analysis/labels']
selected=full.loc[full.status.eq('no_exact_whole_share_pair'),['trade_date','bar_end_at','fill_at','decision_nav','phase','soxl_funded','soxx_funded']].copy()
assert len(selected)==objects['analysis/audit']['counts']['no_exact_whole_share_pair']==50719 and len(selected)<=plan['budget']['maximum_rows']
parent,_=load_verified_recorder_objects(root/p['card'],expected_recorder_id=p['recorder_id'],artifacts=['analysis/native/dated_f696_0'],expected_manifest_sha256=p['manifest_sha256'])
book=parent['analysis/native/dated_f696_0'];tariff=json.loads((root/plan['tariff_source_plan']).read_text())['tariff'];fees=_fee_snapshots(book['independent_replay'],book['orders'],tariff)
print('verified source-only no-exact-fill rows',len(selected))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[31299:MainThread](2026-10-30 14:35:41,361) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31299:MainThread](2026-10-30 14:35:41,765) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31299:MainThread](2026-10-30 14:35:41,765) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


verified source-only no-exact-fill rows 50719


In [2]:
raw=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=raw.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+f.removeprefix('$') for f in x.columns];parts.append(x)
assert parts[0].index.equals(parts[1].index)
quotes=pd.concat(parts,axis=1).sort_index();assert quotes.index.is_unique
print('Qlib bars',len(raw))

Qlib bars 820440


In [3]:
counts={};examples={};max_nav_error=0.;checked=0
for row in selected.itertuples(index=False):
 day=pd.Timestamp(row.trade_date);late=row.phase=='after_1032'
 stamp=day+pd.Timedelta(hours=10,minutes=32) if late else day+pd.Timedelta(hours=9,minutes=47)
 pos=book['native_positions'][stamp]
 q=np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float);cash=float(pos.get_cash())
 mark=quotes.loc[pd.Timestamp(row.bar_end_at)];fill=quotes.loc[pd.Timestamp(row.fill_at)+pd.Timedelta(minutes=1)]
 assert mark.SOXL_price_valid==mark.SOXX_price_valid==fill.SOXL_price_valid==fill.SOXX_price_valid==1
 price=np.array([mark.SOXL_close,mark.SOXX_close],float);nav=float(q@price+cash)
 max_nav_error=max(max_nav_error,abs(nav-float(row.decision_nav)))
 if abs(nav-float(row.decision_nav))>1e-7:raise ValueError('blocked: decision NAV source differs')
 w=np.r_[q*price,cash]/nav
 fill_price=np.array([fill.SOXL_open,fill.SOXX_open],float);raw_fill=np.array([fill.SOXL_raw_open,fill.SOXX_raw_open],float)
 for symbol,change in (('SOXL',np.array([.10/3,0.,-.10/3])),('SOXX',np.array([0.,.10,-.10]))):
  trade=settle_tiered_event(q,cash,desired=w+change,adjusted_prices=fill_price,raw_prices=raw_fill,stamp=row.fill_at,fees=deepcopy(fees[day][1 if late else 0]))
  orders=trade['orders'];exact=bool(len(orders)==1 and orders[0]['symbol']==symbol and not orders[0]['sell'] and trade['cash']>=-1e-8)
  if exact!=bool(getattr(row,symbol.lower()+'_funded')):raise ValueError('blocked: saved funded identity differs')
  shape='zero_order' if not orders else 'exact_one_intended_buy' if exact else 'multi_order' if len(orders)>1 else 'single_wrong_order'
  key=(int(day.year),symbol,shape);counts[key]=counts.get(key,0)+1
  if shape!='exact_one_intended_buy' and shape not in examples:
   examples[shape]={'trade_date':str(day.date()),'bar_end_at':str(row.bar_end_at),'arm':symbol,'orders':[(o['symbol'],'sell' if o['sell'] else 'buy',int(o['raw_shares'])) for o in orders]}
 checked+=1
assert checked==len(selected)
annual=[]
for year in (2018,2019,2020,2021,2022):
 for arm in ('SOXL','SOXX'):
  row={'year':year,'arm':arm,'no_exact_pair_source_rows':int((selected.trade_date.dt.year==year).sum())}
  for shape in plan['order_classes']:row[shape]=counts.get((year,arm,shape),0)
  assert sum(row[shape] for shape in plan['order_classes'])==row['no_exact_pair_source_rows']
  annual.append(row)
aggregate={shape:sum(v for (y,arm,s),v in counts.items() if s==shape) for shape in plan['order_classes']}
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'rows':len(selected),'arms_reconstructed':2*len(selected),'max_decision_nav_error_usd':max_nav_error,'aggregate_arm_order_shapes':aggregate,'annual_arm_shapes':annual,'first_source_examples':examples,'economic_value_columns_accessed':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'rows':audit['rows'],'aggregate_arm_order_shapes':aggregate,'examples':examples},ensure_ascii=False,indent=2))

{
  "rows": 50719,
  "aggregate_arm_order_shapes": {
    "zero_order": 0,
    "exact_one_intended_buy": 50692,
    "single_wrong_order": 0,
    "multi_order": 50746
  },
  "examples": {
    "multi_order": {
      "trade_date": "2018-10-15",
      "bar_end_at": "2018-10-15 11:25:00",
      "arm": "SOXX",
      "orders": [
        [
          "SOXL",
          "sell",
          1
        ],
        [
          "SOXX",
          "buy",
          59
        ]
      ]
    }
  }
}


## 中文结论

SOXX历史门未运行。本只读Qlib审计重放50719条`no_exact_whole_share_pair`来源分钟的101438个资产臂，决策NAV与保存标签最大误差0、保存funded标志逐行完全相同。零订单0、单笔买错资产0；SOXX臂50239次发生多笔订单，SOXL臂507次发生多笔，其他50692个臂正好一笔预定买单。例：2018-10-15 11:25的SOXX目标先额外卖1股SOXL再买59股SOXX。原因是以决策时权重构造完整三资产目标，下一bar价格漂移让原持仓也被目标再平衡；不能把50719行称为“没买到整股”，也不能按事后精确成交条件训练实时触发器。旧标签保留其完整目标权重动作作用域。下一动作版本应在已完成分钟锁定原价整数股数量，下一bar只尝试该资产订单；独立冻结来源、现金/费用、拒单和同状态五会话标签，不能把旧值挪用。2021/2022价值列未访问。